In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, shutil, sys

BASE = '/kaggle/input/datasets/adityaahirwar13/renaissance-ocr-data'
WORK = '/kaggle/working/renaissance_ocr'

for folder in ['data/raw', 'data/ground_truth', 'data/images',
               'data/crops', 'data/rodrigo', 'checkpoints', 'outputs', 'logs']:
    os.makedirs(f'{WORK}/{folder}', exist_ok=True)

for f in os.listdir(BASE):
    if f.endswith(('.py', '.ipynb', '.txt')):
        shutil.copy(f'{BASE}/{f}', f'{WORK}/{f}')

rodrigo_src = f'{BASE}/rodrigo/rodrigo' if os.path.exists(f'{BASE}/rodrigo/rodrigo') else f'{BASE}/rodrigo'
shutil.copytree(rodrigo_src, f'{WORK}/data/rodrigo', dirs_exist_ok=True)

raw_src = f'{BASE}/raw/raw' if os.path.exists(f'{BASE}/raw/raw') else f'{BASE}/raw'
for f in os.listdir(raw_src):
    if os.path.isfile(f'{raw_src}/{f}'):
        shutil.copy(f'{raw_src}/{f}', f'{WORK}/data/raw/{f}')

gt_src = f'{BASE}/ground_truth/ground_truth' if os.path.exists(f'{BASE}/ground_truth/ground_truth') else f'{BASE}/ground_truth'
for f in os.listdir(gt_src):
    if os.path.isfile(f'{gt_src}/{f}'):
        shutil.copy(f'{gt_src}/{f}', f'{WORK}/data/ground_truth/{f}')

os.chdir(WORK)
sys.path.insert(0, WORK)
print(f'PDFs: {len(os.listdir(WORK+"/data/raw"))}')
print(f'GT  : {len(os.listdir(WORK+"/data/ground_truth"))}')
print(f'Rod : {len(os.listdir(WORK+"/data/rodrigo/images"))}')

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers>=4.45.0', 'peft>=0.12.0',
    'bitsandbytes>=0.43.0', 'accelerate>=0.26.0',
    'qwen-vl-utils', 'pymupdf', 'opencv-python-headless',
    'jiwer', 'bert-score', 'nltk'
], capture_output=True)
print('Done.')

In [ ]:
train_code = '''
import os, json, random
from pathlib import Path
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from qwen_vl_utils import process_vision_info
from tqdm import tqdm
import shutil

from config import (
    CHECKPOINTS_DIR, EPOCHS, LEARNING_RATE,
    LOGGING_STEPS, SEED, LORA_ADAPTER,
    TRANSCRIBE_PROMPT, MAX_SEQ_LEN,
)
from dataset import HandwrittenLineDataset, split_dataset
from model import load_base_model, apply_lora

torch.manual_seed(SEED)

# Kaggle output dir — survives session
KAGGLE_OUT = Path('/kaggle/working')


def save_adapter_safely(model, processor, tag="latest"):
    """
    Save adapter to THREE locations:
    1. checkpoints/qwen2vl_lora_renaissance  (normal path)
    2. /kaggle/working/adapter_{tag}         (survives working dir wipe)
    3. zip it immediately                    (downloadable from output panel)
    """
    # location 1
    model.save_pretrained(str(LORA_ADAPTER))
    processor.save_pretrained(str(LORA_ADAPTER))

    # location 2 — root of kaggle working
    out_dir = KAGGLE_OUT / f"adapter_{tag}"
    model.save_pretrained(str(out_dir))
    processor.save_pretrained(str(out_dir))

    # location 3 — zip it
    zip_path = str(KAGGLE_OUT / f"adapter_{tag}")
    shutil.make_archive(zip_path, "zip", str(out_dir))
    size = Path(zip_path + ".zip").stat().st_size / 1e6
    print(f"[save] adapter_{tag}.zip saved ({size:.1f} MB) — download from Output panel")


def process_single_sample(processor, image, gt_text, device):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  TRANSCRIBE_PROMPT},
            ],
        },
        {"role": "assistant", "content": gt_text},
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(device)

    input_ids = inputs["input_ids"][0]
    labels    = input_ids.clone()

    prompt_text = processor.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True
    )
    prompt_len = len(processor.tokenizer(
        prompt_text, truncation=True, max_length=MAX_SEQ_LEN
    ).input_ids)
    labels[:prompt_len] = -100
    inputs["labels"] = labels.unsqueeze(0)
    return inputs


def train():
    print("=" * 60)
    print(" RenAIssance HTR — Training (Dual T4 + Safe Saves)")
    print("=" * 60)

    # use both GPUs via DataParallel for forward pass
    # but keep LoRA on primary GPU for backward
    n_gpus = torch.cuda.device_count()
    print(f"[train] GPUs available: {n_gpus}")
    device = torch.device("cuda:0")

    # load model on GPU 0
    model, processor = load_base_model(quantize=True)
    model = apply_lora(model)
    model.train()

    # dataset
    full_dataset = HandwrittenLineDataset(augment=True)
    train_ds, _  = split_dataset(full_dataset, val_ratio=0.0)
    print(f"[train] Samples: {len(train_ds)}")

    # optimizer
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
    )
    total_steps = (len(train_ds) // 8) * EPOCHS
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps  = 10,
        num_training_steps = total_steps,
    )

    history    = {"train_loss": [], "steps": []}
    global_step = 0
    GRAD_ACCUM  = 8

    for epoch in range(EPOCHS):
        print(f"\\n[train] Epoch {epoch+1}/{EPOCHS}")
        epoch_loss = 0.0
        valid_count = 0
        optimizer.zero_grad()

        indices = list(range(len(train_ds)))
        random.shuffle(indices)

        for i, idx in enumerate(tqdm(indices, desc=f"Epoch {epoch+1}")):
            sample  = train_ds[idx]
            try:
                inputs  = process_single_sample(
                    processor, sample["image"], sample["text"], device
                )
                outputs = model(**inputs)
                loss    = outputs.loss / GRAD_ACCUM

                if loss is None or torch.isnan(loss):
                    continue

                loss.backward()
                epoch_loss  += loss.item() * GRAD_ACCUM
                valid_count += 1

            except Exception:
                continue

            if (i + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % LOGGING_STEPS == 0:
                    avg = epoch_loss / max(valid_count, 1)
                    print(f"  step={global_step}  loss={avg:.4f}")
                    history["train_loss"].append(avg)
                    history["steps"].append(global_step)

        avg = epoch_loss / max(valid_count, 1)
        print(f"[train] Epoch {epoch+1} avg loss: {avg:.4f}")

        # SAVE AFTER EVERY EPOCH — guaranteed
        print(f"[train] Saving epoch {epoch+1} adapter ...")
        save_adapter_safely(model, processor, tag=f"epoch{epoch+1}")
        print(f"[train] Epoch {epoch+1} saved. Download adapter_epoch{epoch+1}.zip NOW from Output panel.")

    # final save
    save_adapter_safely(model, processor, tag="final")

    hist_path = CHECKPOINTS_DIR / "training_history.json"
    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2)
    print("[train] Done.")
    return history
'''

with open('/kaggle/working/renaissance_ocr/train.py', 'w') as f:
    f.write(train_code)
print('train.py written.')

In [ ]:
import sys, os
os.chdir('/kaggle/working/renaissance_ocr')
sys.path.insert(0, '/kaggle/working/renaissance_ocr')

from dataset import convert_all_pdfs, load_all_gt, build_line_gt_map
convert_all_pdfs()
gt_map  = load_all_gt()
flat_gt = build_line_gt_map(gt_map)
print(f'GT lines: {len(flat_gt)}')

In [ ]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # use both T4s
os.chdir('/kaggle/working/renaissance_ocr')
sys.path.insert(0, '/kaggle/working/renaissance_ocr')

for mod in list(sys.modules.keys()):
    if mod in ['train', 'model', 'dataset', 'config']:
        del sys.modules[mod]

from train import train
history = train()

In [1]:
import os, shutil, sys

BASE = '/kaggle/input/datasets/adityaahirwar13/renaissance-ocr-data'
EPOCH2 = '/kaggle/input/datasets/adityaahirwar13/epoch2'
WORK = '/kaggle/working/renaissance_ocr'

# create folders
for folder in ['data/raw', 'data/ground_truth', 'data/images',
               'data/crops', 'data/rodrigo', 'checkpoints', 'outputs', 'logs']:
    os.makedirs(f'{WORK}/{folder}', exist_ok=True)

# copy python files
for f in os.listdir(BASE):
    if f.endswith(('.py', '.ipynb', '.txt')):
        shutil.copy(f'{BASE}/{f}', f'{WORK}/{f}')

# rodrigo
rodrigo_src = f'{BASE}/rodrigo/rodrigo' if os.path.exists(f'{BASE}/rodrigo/rodrigo') else f'{BASE}/rodrigo'
shutil.copytree(rodrigo_src, f'{WORK}/data/rodrigo', dirs_exist_ok=True)

# raw PDFs
raw_src = f'{BASE}/raw/raw' if os.path.exists(f'{BASE}/raw/raw') else f'{BASE}/raw'
for f in os.listdir(raw_src):
    if os.path.isfile(f'{raw_src}/{f}'):
        shutil.copy(f'{raw_src}/{f}', f'{WORK}/data/raw/{f}')

# ground truth
gt_src = f'{BASE}/ground_truth/ground_truth' if os.path.exists(f'{BASE}/ground_truth/ground_truth') else f'{BASE}/ground_truth'
for f in os.listdir(gt_src):
    if os.path.isfile(f'{gt_src}/{f}'):
        shutil.copy(f'{gt_src}/{f}', f'{WORK}/data/ground_truth/{f}')

# load epoch2 adapter — check structure
print('Epoch2 dataset contents:')
for root, dirs, files in os.walk(EPOCH2):
    for f in files:
        print(' ', os.path.join(root, f).replace(EPOCH2, ''))

Epoch2 dataset contents:
  /adapter_model.safetensors
  /adapter_config.json
  /README.md
  /tokenizer.json
  /tokenizer_config.json
  /chat_template.jinja
  /processor_config.json


In [2]:
import shutil, os, subprocess

EPOCH2 = '/kaggle/input/datasets/adityaahirwar13/epoch2'
WORK   = '/kaggle/working/renaissance_ocr'

# copy epoch2 adapter to correct location
adapter_dst = f'{WORK}/checkpoints/qwen2vl_lora_renaissance'
os.makedirs(adapter_dst, exist_ok=True)
shutil.copytree(EPOCH2, adapter_dst, dirs_exist_ok=True)

print('Adapter copied. Contents:')
for f in os.listdir(adapter_dst):
    print(f'  {f}')

# install dependencies
subprocess.run(['pip', 'install', '-q',
    'transformers>=4.45.0', 'peft>=0.12.0',
    'bitsandbytes>=0.43.0', 'accelerate>=0.26.0',
    'qwen-vl-utils', 'pymupdf', 'opencv-python-headless',
], capture_output=True)
print('Dependencies installed.')

Adapter copied. Contents:
  tokenizer_config.json
  chat_template.jinja
  tokenizer.json
  processor_config.json
  adapter_model.safetensors
  adapter_config.json
  README.md
Dependencies installed.


In [3]:
import sys, os
os.chdir('/kaggle/working/renaissance_ocr')
sys.path.insert(0, '/kaggle/working/renaissance_ocr')

from dataset import convert_all_pdfs, load_all_gt, build_line_gt_map
convert_all_pdfs()
gt_map  = load_all_gt()
flat_gt = build_line_gt_map(gt_map)
print(f'GT lines: {len(flat_gt)}')

[config] env=kaggle  device=cuda
[dataset] Skip source4.pdf — already converted.
[dataset] Skip source2.pdf — already converted.
[dataset] Skip source3.pdf — already converted.
[dataset] Skip source1.pdf — already converted.
[dataset] Skip source5.pdf — already converted.
[dataset]  source4.docx: 1 page(s) GT, 25 lines, 8 paleographer notes
[dataset]  source1.docx: 1 page(s) GT, 24 lines, 8 paleographer notes
[dataset]  source3.docx: 1 page(s) GT, 21 lines, 8 paleographer notes
[dataset]  source5.docx: 1 page(s) GT, 24 lines, 8 paleographer notes
[dataset]  source2.docx: 1 page(s) GT, 33 lines, 8 paleographer notes
[dataset] GT loaded for 5 source(s).
GT lines: 127


In [4]:
resume_code = '''
import os, json, random
from pathlib import Path
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, AutoProcessor
from qwen_vl_utils import process_vision_info
from tqdm import tqdm
import shutil
from peft import PeftModel

from config import (
    CHECKPOINTS_DIR, LEARNING_RATE,
    LOGGING_STEPS, SEED, LORA_ADAPTER,
    TRANSCRIBE_PROMPT, MAX_SEQ_LEN,
)
from dataset import HandwrittenLineDataset, split_dataset
from model import load_base_model

torch.manual_seed(SEED)

KAGGLE_OUT   = Path("/kaggle/working")
START_EPOCH  = 2   # already done
RESUME_EPOCHS = 3  # run epochs 3, 4, 5


def save_adapter_safely(model, processor, tag):
    model.save_pretrained(str(LORA_ADAPTER))
    processor.save_pretrained(str(LORA_ADAPTER))
    out_dir  = KAGGLE_OUT / f"adapter_{tag}"
    model.save_pretrained(str(out_dir))
    processor.save_pretrained(str(out_dir))
    zip_path = str(KAGGLE_OUT / f"adapter_{tag}")
    shutil.make_archive(zip_path, "zip", str(out_dir))
    size = Path(zip_path + ".zip").stat().st_size / 1e6
    print(f"[save] adapter_{tag}.zip ({size:.1f} MB)")
    print(f">>> DOWNLOAD adapter_{tag}.zip FROM OUTPUT PANEL NOW <<<")


def process_single_sample(processor, image, gt_text, device):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  TRANSCRIBE_PROMPT},
            ],
        },
        {"role": "assistant", "content": gt_text},
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(device)

    input_ids  = inputs["input_ids"][0]
    labels     = input_ids.clone()
    prompt_len = len(processor.tokenizer(
        processor.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True
        ),
        truncation=True, max_length=MAX_SEQ_LEN
    ).input_ids)
    labels[:prompt_len] = -100
    inputs["labels"]    = labels.unsqueeze(0)
    return inputs


def resume_train():
    print("=" * 60)
    print(f" Resuming from epoch {START_EPOCH+1} to 5")
    print("=" * 60)

    device = torch.device("cuda:0")

    # load base + epoch2 adapter
    base_model, processor = load_base_model(quantize=True)
    print(f"[train] Loading epoch {START_EPOCH} adapter...")
    model = PeftModel.from_pretrained(base_model, str(LORA_ADAPTER))
    model.train()

    # make sure LoRA params are trainable
    for name, param in model.named_parameters():
        if "lora" in name.lower():
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[train] Trainable params: {trainable:,}")

    # dataset
    full_dataset = HandwrittenLineDataset(augment=True)
    train_ds, _  = split_dataset(full_dataset, val_ratio=0.0)
    print(f"[train] Samples: {len(train_ds)}")

    # optimizer with half LR — already partially trained
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE * 0.5,
    )
    total_steps = (len(train_ds) // 8) * RESUME_EPOCHS
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = 5,
        num_training_steps = total_steps,
    )

    history     = {"train_loss": [], "steps": []}
    global_step = 0
    GRAD_ACCUM  = 8

    for epoch in range(RESUME_EPOCHS):
        actual_epoch = START_EPOCH + epoch + 1
        print(f"\\n[train] Epoch {actual_epoch}/5")
        epoch_loss  = 0.0
        valid_count = 0
        optimizer.zero_grad()

        indices = list(range(len(train_ds)))
        random.shuffle(indices)

        for i, idx in enumerate(tqdm(indices, desc=f"Epoch {actual_epoch}")):
            sample = train_ds[idx]
            try:
                inputs  = process_single_sample(
                    processor, sample["image"], sample["text"], device
                )
                outputs = model(**inputs)
                loss    = outputs.loss / GRAD_ACCUM

                if loss is None or torch.isnan(loss):
                    continue

                loss.backward()
                epoch_loss  += loss.item() * GRAD_ACCUM
                valid_count += 1

            except Exception:
                continue

            if (i + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % LOGGING_STEPS == 0:
                    avg = epoch_loss / max(valid_count, 1)
                    print(f"  step={global_step}  loss={avg:.4f}")
                    history["train_loss"].append(avg)
                    history["steps"].append(global_step)

        avg = epoch_loss / max(valid_count, 1)
        print(f"[train] Epoch {actual_epoch} avg loss: {avg:.4f}")
        save_adapter_safely(model, processor, tag=f"epoch{actual_epoch}")

    save_adapter_safely(model, processor, tag="final")
    print("[train] Resume complete.")
    return history
'''

with open('/kaggle/working/renaissance_ocr/resume_train.py', 'w') as f:
    f.write(resume_code)
print('resume_train.py written.')

resume_train.py written.


In [ ]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.chdir('/kaggle/working/renaissance_ocr')
sys.path.insert(0, '/kaggle/working/renaissance_ocr')

for mod in list(sys.modules.keys()):
    if mod in ['train', 'resume_train', 'model', 'dataset', 'config']:
        del sys.modules[mod]

from resume_train import resume_train
history = resume_train()


[config] env=kaggle  device=cuda
 Resuming from epoch 3 to 5
[model] Loading Qwen/Qwen2-VL-2B-Instruct …


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

[model] Base model loaded. Parameters: 1.22B
[train] Loading epoch 2 adapter...
[train] Trainable params: 18,464,768
[dataset]  source4.docx: 1 page(s) GT, 25 lines, 8 paleographer notes
[dataset]  source1.docx: 1 page(s) GT, 24 lines, 8 paleographer notes
[dataset]  source3.docx: 1 page(s) GT, 21 lines, 8 paleographer notes
[dataset]  source5.docx: 1 page(s) GT, 24 lines, 8 paleographer notes
[dataset]  source2.docx: 1 page(s) GT, 33 lines, 8 paleographer notes
[dataset] GT loaded for 5 source(s).
[dataset] RenAIssance GT crops matched: 0
[dataset] Rodrigo train split: 9000 stems
[dataset] Rodrigo transcriptions loaded: 15010
[dataset] Rodrigo: 9000 train samples loaded. (0 images missing — normal if small)
[dataset] Total samples: 9000
[dataset] Train: 9000 (-1 renaissance + 9000 rodrigo)
[dataset] Val:   0 (renaissance only)
[train] Samples: 9000

[train] Epoch 3/5


...

Epoch 4:  45%|████▍     | 4040/9000 [43:28<54:20,  1.52it/s]  

  step=1630  loss=0.0118


Epoch 4:  46%|████▌     | 4120/9000 [44:20<53:23,  1.52it/s]

  step=1640  loss=0.0118


Epoch 4:  46%|████▌     | 4149/9000 [44:39<52:47,  1.53it/s]